# Comprehensive EDA


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
import cv2
from collections import Counter
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Success")

## Define Dataset Path

In [ ]:
# Base Path
BASE_PATH = 'chest_xray'

# Define Paths for train, test, and validation sets
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
TEST_PATH  = os.path.join(BASE_PATH, 'test')
VAL_PATH   = os.path.join(BASE_PATH, 'val')

CLASSES = ['NORMAL', 'PNEUMONIA']

print(f"base path : {BASE_PATH}")
print(f"Train path : {TRAIN_PATH}")
print(f"Test path : {TEST_PATH}")
print(f"Validation path : {VAL_PATH}")
print(f"CLASSES   : {CLASSES}")


## NOW VERIFY directory structure
> Check if all directories exist and are accessible


In [ ]:
def verify_directories():
    
    paths = {
       'Train': TRAIN_PATH,
       'Test' : TEST_PATH ,
       'Val'  : VAL_PATH 
    }

    for name, path in paths.items():
        if os.path.exists(path):
            print("Working")

        else:
            print("Ther is something wrong")

    return all(os.path.exists(path) for path in paths.values())

all_exist = verify_directories()
if all_exist:
    print('Workig')

## Count Images in each directory
> Get the distribution of images across train / test / val splits

In [ ]:
def count_images(directory):
    counts = {}
    for class_name in CLASSES:
        class_path = os.path.join(directory, class_name)
        if os.path.exists(class_path):
            image_files  = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpeg', '.png', '.jpg'))]
            counts[class_name] = len(image_files)
        else:
            counts[class_name] = 0

    return counts

train_counts = count_images(TRAIN_PATH)
test_counts  = count_images(TEST_PATH)
val_counts   = count_images(VAL_PATH)

print("="*60)
print("IMAGE DISTRIBUTION ACROSS SPLITS")
print("="*60)
print("\nTRAIN SET:")
print(f"  NORMAL: {train_counts['NORMAL']} images")
print(f"  PNEUMONIA: {train_counts['PNEUMONIA']} images")
print(f"  Total: {sum(train_counts.values())} images")

print("\nTEST SET:")
print(f"  NORMAL: {test_counts['NORMAL']} images")
print(f"  PNEUMONIA: {test_counts['PNEUMONIA']} images")
print(f"  Total: {sum(test_counts.values())} images")

print("\nVALIDATION SET:")
print(f"  NORMAL: {val_counts['NORMAL']} images")
print(f"  PNEUMONIA: {val_counts['PNEUMONIA']} images")
print(f"  Total: {sum(val_counts.values())} images")

print(f"\n{'='*60}")
print(f"TOTAL DATASET SIZE: {sum(train_counts.values()) + sum(test_counts.values()) + sum(val_counts.values())} images")
print(f"{'='*60}")

## Distribution DataFrame
> Organize the counts into a structured DataFrame for Analysis

In [ ]:
# Create a DataFrame
# We know that we have Train/test/val folders. Each Folder has two folders
# One NORMAL and ONE Pnuemonia

distribution_data = {
    'Split' : ['Train', 'Train', 'Test', 'Test', 'Validation', 'Validation'],
    'Class' : ['NORMAL', 'PNEUMONIA', 'NORMAL', 'PNEUMONIA', 'NORMAL', 'PNEUMONIA'],
    'Count' : [
        train_counts['NORMAL'], train_counts['PNEUMONIA'],
        test_counts['NORMAL'], test_counts['PNEUMONIA'],
        val_counts['NORMAL'], val_counts['PNEUMONIA'],
        
    ]
}

df= pd.DataFrame(distribution_data)
print("\nDataset Distribution DataFrame : ")
print(df)
# Calculate percentages
df['Percentage'] = (df['Count'] / df.groupby('Split')['Count'].transform('sum') * 100).round(2)
print("\nWith Percentages:")
print(df)

## Visualize Class Distribution
> Create Bar plots to visualize Class Imbalance

In [ ]:
fig, axes = plt.subplots(1,3, figsize = (18, 5))

splits = ['Train', 'Test', 'Validation']
split_data = [train_counts, test_counts, val_counts]

for idx, (split_name, counts) in enumerate(zip(splits, split_data)):
    ax = axes[idx]
    bars = ax.bar(counts.keys(), counts.values(), color=['#3498db', '#e74c3c'], alpha=0.7, edgecolor='black')
    
    # Adding value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
        
    ax.set_title(f'{split_name} Set Distribution', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Images', fontsize=12)
    ax.set_xlabel('Class', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('class_distribution_bars.png', dpi=300, bbox_inches='tight')
plt.show()
print("Bar plot saved as 'class_distribution_bars.png'")


## Class Distribution Pie Chart
> Show proportions using pie charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#3498db', '#e74c3c']
explode = (0.05, 0.05)

for idx, (split_name, counts) in enumerate(zip(splits, split_data)):
    ax = axes[idx]
    wedges, texts, autotexts = ax.pie(
        counts.values(),
        labels=counts.keys(),
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        explode=explode,
        shadow=True
    )
    
    # Beautify text
    for text in texts:
        text.set_fontsize(12)
        text.set_fontweight('bold')
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(11)
        autotext.set_fontweight('bold')
    
    ax.set_title(f'{split_name} Set Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution_pies.png', dpi=300, bbox_inches='tight')
plt.show()

print("Pie charts saved as 'class_distribution_pies.png'")

## Calculate Class Imbalance Ratio
> Quantify the Imbalance in the Dataset

In [ ]:
def calculate_imbalance_ratio(counts):
    max_count = max(counts.values())
    min_count = min(counts.values())

    return max_count / min_count if min_count > 0 else float('inf')

print('^'*60)
print("Class Imbalance analysis")
print('^'*60) 

train_ratio = calculate_imbalance_ratio(train_counts)
test_ratio = calculate_imbalance_ratio(test_counts)
val_ratio = calculate_imbalance_ratio(val_counts)

print(f"\nTrain Set Imbalance Ratio: {train_ratio:.2f}:1")
print(f"Test Set Imbalance Ratio: {test_ratio:.2f}:1")
print(f"Validation Set Imbalance Ratio: {val_ratio:.2f}:1")

print("\nInterpretation:")
if train_ratio > 2:
    print("Significant class imbalance detected in training set!")
else:
    print("Training set is relatively balanced")

## Collect Image MetaData
> Extract detailed Information about each Image (size, dimensions, etc.)

In [ ]:
def collect_metadata(directory, class_name, split_name, max_images=None):
    metadata = []
    class_path = os.path.join(directory, class_name)

    if not os.path.exists(class_path):
        return metadata
    image_files = [f for f in os.listdir(class_path) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if max_images:
        image_files = image_files[:max_images]
    
    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        try:
            img = Image.open(img_path)
            width, height = img.size
            
            # Get file size
            file_size = os.path.getsize(img_path) / 1024  # in KB
            
            metadata.append({
                'filename': img_file,
                'split': split_name,
                'class': class_name,
                'width': width,
                'height': height,
                'aspect_ratio': width / height,
                'total_pixels': width * height,
                'file_size_kb': file_size,
                'format': img.format
            })
        except Exception as e:
            print(f"Error processing {img_file}: {e}")
    
    return metadata

print("Collecting image metadata...")
print("This may take a few minutes...\n")

# Collect metadata from all splits (limiting to prevent memory issues)
all_metadata = []

all_metadata.extend(collect_metadata(TRAIN_PATH, 'NORMAL', 'Train', max_images=500))
all_metadata.extend(collect_metadata(TRAIN_PATH, 'PNEUMONIA', 'Train', max_images=500))
all_metadata.extend(collect_metadata(TEST_PATH, 'NORMAL', 'Test'))
all_metadata.extend(collect_metadata(TEST_PATH, 'PNEUMONIA', 'Test'))
all_metadata.extend(collect_metadata(VAL_PATH, 'NORMAL', 'Validation'))
all_metadata.extend(collect_metadata(VAL_PATH, 'PNEUMONIA', 'Validation'))

df_metadata = pd.DataFrame(all_metadata)
print(f"Collected metadata for {len(df_metadata)} images")
print(f"\nDataFrame shape: {df_metadata.shape}")
print("\nFirst few rows:")
df_metadata.head()


## Statistical Summary of Image Dimensions
> Understand The Distribution of image sizes

In [ ]:
print("&"*60)
print("IMAGE DIMENSIONS STATISTICS")
print("&"*60)

print("\nOverall Statistics:")
print(df_metadata[['width', 'height', 'aspect_ratio', 'total_pixels', 'file_size_kb']].describe())

print("\n" + "uwu"*60)
print("Statistics by Class:")
print("uwu"*60)

for class_name in CLASSES:
    class_data = df_metadata[df_metadata['class'] == class_name]
    print(f"\n{class_name}:")
    print(f"  Average Width: {class_data['width'].mean():.2f} px")
    print(f"  Average Height: {class_data['height'].mean():.2f} px")
    print(f"  Average Aspect Ratio: {class_data['aspect_ratio'].mean():.2f}")
    print(f"  Average File Size: {class_data['file_size_kb'].mean():.2f} KB")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Width distribution
ax1 = axes[0, 0]
for class_name in CLASSES:
    class_data = df_metadata[df_metadata['class'] == class_name]
    ax1.hist(class_data['width'], bins=30, alpha=0.6, label=class_name, edgecolor='black')
ax1.set_xlabel('Width (pixels)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Image Widths', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Height distribution
ax2 = axes[0, 1]
for class_name in CLASSES:
    class_data = df_metadata[df_metadata['class'] == class_name]
    ax2.hist(class_data['height'], bins=30, alpha=0.6, label=class_name, edgecolor='black')
ax2.set_xlabel('Height (pixels)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Image Heights', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Aspect ratio distribution
ax3 = axes[1, 0]
for class_name in CLASSES:
    class_data = df_metadata[df_metadata['class'] == class_name]
    ax3.hist(class_data['aspect_ratio'], bins=30, alpha=0.6, label=class_name, edgecolor='black')
ax3.set_xlabel('Aspect Ratio (width/height)', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.set_title('Distribution of Aspect Ratios', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# Width vs Height scatter
ax4 = axes[1, 1]
for class_name, color in zip(CLASSES, ['#3498db', '#e74c3c']):
    class_data = df_metadata[df_metadata['class'] == class_name]
    ax4.scatter(class_data['width'], class_data['height'], 
               alpha=0.5, label=class_name, s=20, color=color)
ax4.set_xlabel('Width (pixels)', fontsize=12)
ax4.set_ylabel('Height (pixels)', fontsize=12)
ax4.set_title('Width vs Height Scatter Plot', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('image_dimensions_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Dimension analysis saved as 'image_dimensions_analysis.png'")

## File size analysis
> Understand storage requirements

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# File size distribution
ax1 = axes[0]
for class_name in CLASSES:
    class_data = df_metadata[df_metadata['class'] == class_name]
    ax1.hist(class_data['file_size_kb'], bins=30, alpha=0.6, 
            label=class_name, edgecolor='black')
ax1.set_xlabel('File Size (KB)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of File Sizes', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot of file sizes by class
ax2 = axes[1]
data_to_plot = [df_metadata[df_metadata['class'] == class_name]['file_size_kb'] 
                for class_name in CLASSES]
bp = ax2.boxplot(data_to_plot, labels=CLASSES, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('File Size (KB)', fontsize=12)
ax2.set_title('File Size Distribution by Class', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('file_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("File size analysis saved as 'file_size_analysis.png'")


## Load and Analyze sample images
> Analyze Pixel Intensity distributions

In [ ]:
def load_sample_images(directory, class_name, n_samples=5):
    """Load sample images from a class"""
    class_path = os.path.join(directory, class_name)
    image_files = [f for f in os.listdir(class_path) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:n_samples]
    
    images = []
    for img_file in image_files:
        img_path = os.path.join(class_path, img_file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img)
    
    return images

print("Loading sample images for pixel analysis...")

# Load samples from training set
normal_samples = load_sample_images(TRAIN_PATH, 'NORMAL', n_samples=20)
pneumonia_samples = load_sample_images(TRAIN_PATH, 'PNEUMONIA', n_samples=20)

print(f"Loaded {len(normal_samples)} NORMAL samples")
print(f"Loaded {len(pneumonia_samples)} PNEUMONIA samples")


## Pixel Intensity values
> Compare pixel intensities between classes


In [ ]:
normal_intensities = [img.flatten() for img in normal_samples]
pneumonia_intensities = [img.flatten() for img in pneumonia_samples]

normal_mean_intensity = np.mean([np.mean(img) for img in normal_intensities])
pneumonia_mean_intensity = np.mean([np.mean(img) for img in pneumonia_intensities])

print("="*60)
print("PIXEL INTENSITY ANALYSIS")
print("="*60)
print(f"\nNORMAL images - Mean intensity: {normal_mean_intensity:.2f}")
print(f"PNEUMONIA images - Mean intensity: {pneumonia_mean_intensity:.2f}")
print(f"Difference: {abs(normal_mean_intensity - pneumonia_mean_intensity):.2f}")

# Plot intensity distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
for intensities in normal_intensities:
    ax1.hist(intensities, bins=50, alpha=0.3, color='blue', edgecolor='none')
ax1.set_xlabel('Pixel Intensity', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('NORMAL - Pixel Intensity Distribution', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

ax2 = axes[1]
for intensities in pneumonia_intensities:
    ax2.hist(intensities, bins=50, alpha=0.3, color='red', edgecolor='none')
ax2.set_xlabel('Pixel Intensity', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('PNEUMONIA - Pixel Intensity Distribution', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pixel_intensity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Intensity distribution saved as 'pixel_intensity_distribution.png'")

## Display sample imgaes grid
> Visualize actual X-ray Images from both classes

In [ ]:
def display_sample_grid(normal_imgs, pneumonia_imgs, n_samples=5):
    """Display a grid of sample images"""
    fig, axes = plt.subplots(2, n_samples, figsize=(20, 8))
    
    # Display NORMAL samples
    for idx in range(n_samples):
        ax = axes[0, idx]
        if idx < len(normal_imgs):
            ax.imshow(normal_imgs[idx], cmap='gray')
            ax.set_title('NORMAL', fontsize=12, fontweight='bold', color='blue')
        ax.axis('off')
    
    # Display PNEUMONIA samples
    for idx in range(n_samples):
        ax = axes[1, idx]
        if idx < len(pneumonia_imgs):
            ax.imshow(pneumonia_imgs[idx], cmap='gray')
            ax.set_title('PNEUMONIA', fontsize=12, fontweight='bold', color='red')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('sample_images_grid.png', dpi=300, bbox_inches='tight')
    plt.show()

display_sample_grid(normal_samples[:5], pneumonia_samples[:5])
print("Sample images grid saved as 'sample_images_grid.png'")


## Image Statistics - Mean and Standard deviation
> Calculate statistical measures from image analysis

In [ ]:
def calculate_image_stats(images):
    """Calculate mean and std for a set of images"""
    means = [np.mean(img) for img in images]
    stds = [np.std(img) for img in images]
    return means, stds

normal_means, normal_stds = calculate_image_stats(normal_samples)
pneumonia_means, pneumonia_stds = calculate_image_stats(pneumonia_samples)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mean comparison
ax1 = axes[0]
ax1.boxplot([normal_means, pneumonia_means], labels=['NORMAL', 'PNEUMONIA'])
ax1.set_ylabel('Mean Pixel Intensity', fontsize=12)
ax1.set_title('Mean Pixel Intensity Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Std comparison
ax2 = axes[1]
ax2.boxplot([normal_stds, pneumonia_stds], labels=['NORMAL', 'PNEUMONIA'])
ax2.set_ylabel('Standard Deviation', fontsize=12)
ax2.set_title('Pixel Intensity Variability Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('image_statistics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Statistics comparison saved as 'image_statistics_comparison.png'")

## Edge Detection Analysis
> Analyze edge content in x-rays using Sobel operator

In [ ]:
def calculate_edge_density(image):
    """Calculate edge density using Sobel operator"""
    sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    edge_magnitude = np.sqrt(sobelx**2 + sobely**2)
    return np.mean(edge_magnitude)

print("Calculating edge densities...")

normal_edge_densities = [calculate_edge_density(img) for img in normal_samples]
pneumonia_edge_densities = [calculate_edge_density(img) for img in pneumonia_samples]

print(f"\nNORMAL - Average edge density: {np.mean(normal_edge_densities):.2f}")
print(f"PNEUMONIA - Average edge density: {np.mean(pneumonia_edge_densities):.2f}")

# Visualize edge densities
fig, ax = plt.subplots(figsize=(10, 6))
data_to_plot = [normal_edge_densities, pneumonia_edge_densities]
bp = ax.boxplot(data_to_plot, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Edge Density', fontsize=12)
ax.set_title('Edge Density Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.savefig('edge_density_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Edge density analysis saved as 'edge_density_analysis.png'")


## Display Edge Detection Examples
> Show edge detection applied to sample images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx in range(2):
    # Original NORMAL
    ax = axes[0, idx*2]
    ax.imshow(normal_samples[idx], cmap='gray')
    ax.set_title(f'NORMAL {idx+1} - Original', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Edges NORMAL
    ax = axes[0, idx*2 + 1]
    sobelx = cv2.Sobel(normal_samples[idx], cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(normal_samples[idx], cv2.CV_64F, 0, 1, ksize=3)
    edges = np.sqrt(sobelx**2 + sobely**2)
    ax.imshow(edges, cmap='hot')
    ax.set_title(f'NORMAL {idx+1} - Edges', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Original PNEUMONIA
    ax = axes[1, idx*2]
    ax.imshow(pneumonia_samples[idx], cmap='gray')
    ax.set_title(f'PNEUMONIA {idx+1} - Original', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Edges PNEUMONIA
    ax = axes[1, idx*2 + 1]
    sobelx = cv2.Sobel(pneumonia_samples[idx], cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(pneumonia_samples[idx], cv2.CV_64F, 0, 1, ksize=3)
    edges = np.sqrt(sobelx**2 + sobely**2)
    ax.imshow(edges, cmap='hot')
    ax.set_title(f'PNEUMONIA {idx+1} - Edges', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('edge_detection_examples.png', dpi=300, bbox_inches='tight')
plt.show()

print("Edge detection examples saved as 'edge_detection_examples.png'")

## Histogram Equalization
> Apply histogram equalization to enhance contrast

In [ ]:
def apply_histogram_equalization(image):
    """Apply histogram equalization"""
    return cv2.equalizeHist(image)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx in range(2):
    # Original NORMAL
    ax = axes[0, idx*2]
    ax.imshow(normal_samples[idx], cmap='gray')
    ax.set_title(f'NORMAL {idx+1} - Original', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Equalized NORMAL
    ax = axes[0, idx*2 + 1]
    equalized = apply_histogram_equalization(normal_samples[idx])
    ax.imshow(equalized, cmap='gray')
    ax.set_title(f'NORMAL {idx+1} - Equalized', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Original PNEUMONIA
    ax = axes[1, idx*2]
    ax.imshow(pneumonia_samples[idx], cmap='gray')
    ax.set_title(f'PNEUMONIA {idx+1} - Original', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Equalized PNEUMONIA
    ax = axes[1, idx*2 + 1]
    equalized = apply_histogram_equalization(pneumonia_samples[idx])
    ax.imshow(equalized, cmap='gray')
    ax.set_title(f'PNEUMONIA {idx+1} - Equalized', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('histogram_equalization.png', dpi=300, bbox_inches='tight')
plt.show()

print("Histogram equalization examples saved as 'histogram_equalization.png'")

## Image Brightness Anaysis
> Analyze Brightness level across classes

In [ ]:
def calculate_brightness(image):
    """Calculate average brightness of an image"""
    return np.mean(image)

normal_brightness = [calculate_brightness(img) for img in normal_samples]
pneumonia_brightness = [calculate_brightness(img) for img in pneumonia_samples]

print("="*60)
print("BRIGHTNESS ANALYSIS")
print("="*60)
print(f"\nNORMAL images:")
print(f"  Mean brightness: {np.mean(normal_brightness):.2f}")
print(f"  Std brightness: {np.std(normal_brightness):.2f}")

print(f"\nPNEUMONIA images:")
print(f"  Mean brightness: {np.mean(pneumonia_brightness):.2f}")
print(f"  Std brightness: {np.std(pneumonia_brightness):.2f}")

# Visualize brightness distribution
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(normal_brightness, bins=20, alpha=0.6, label='NORMAL', 
        color='#3498db', edgecolor='black')
ax.hist(pneumonia_brightness, bins=20, alpha=0.6, label='PNEUMONIA', 
        color='#e74c3c', edgecolor='black')
ax.set_xlabel('Average Brightness', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Image Brightness Distribution Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.savefig('brightness_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Brightness distribution saved as 'brightness_distribution.png'")

## Constract Analysis
> Measure Contrast differences between classes

In [ ]:
def calculate_contrast(image):
    """Calculate contrast using standard deviation"""
    return np.std(image)

normal_contrast = [calculate_contrast(img) for img in normal_samples]
pneumonia_contrast = [calculate_contrast(img) for img in pneumonia_samples]

print("\n" + "="*60)
print("CONTRAST ANALYSIS")
print("="*60)
print("\nNORMAL images:")
print(f"  Mean contrast: {np.mean(normal_contrast):.2f}")
print(f"  Std contrast: {np.std(normal_contrast):.2f}")

print("\nPNEUMONIA images:")
print(f"  Mean contrast: {np.mean(pneumonia_contrast):.2f}")
print(f"  Std contrast: {np.std(pneumonia_contrast):.2f}")

# Visualize contrast comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
ax1 = axes[0]
ax1.hist(normal_contrast, bins=20, alpha=0.6, label='NORMAL', 
         color='#3498db', edgecolor='black')
ax1.hist(pneumonia_contrast, bins=20, alpha=0.6, label='PNEUMONIA', 
         color='#e74c3c', edgecolor='black')
ax1.set_xlabel('Contrast (Std Dev)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Image Contrast Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Box plot
ax2 = axes[1]
data_to_plot = [normal_contrast, pneumonia_contrast]
bp = ax2.boxplot(data_to_plot, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('Contrast (Std Dev)', fontsize=12)
ax2.set_title('Contrast Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('contrast_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Contrast analysis saved as 'contrast_analysis.png'")



## Texture Analysis using GLCM (Grey-level co-occurence Matrix)
> Analyze texture patterns in X-ray images


In [ ]:
from skimage.feature import graycomatrix, graycoprops

def calculate_texture_features(image):
    """Calculate texture features using GLCM"""
    # Normalize to 0-255 range and convert to uint8
    img_normalized = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
    
    # Calculate GLCM
    glcm = graycomatrix(img_normalized, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                       levels=256, symmetric=True, normed=True)
    
    # Extract properties
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    
    return {
        'contrast': contrast,
        'dissimilarity': dissimilarity,
        'homogeneity': homogeneity,
        'energy': energy,
        'correlation': correlation
    }

print("Calculating texture features...")
print("This may take a moment...\n")

normal_textures = [calculate_texture_features(img) for img in normal_samples]
pneumonia_textures = [calculate_texture_features(img) for img in pneumonia_samples]

# Convert to DataFrame for easier analysis
df_normal_texture = pd.DataFrame(normal_textures)
df_pneumonia_texture = pd.DataFrame(pneumonia_textures)

print("="*60)
print("TEXTURE FEATURES SUMMARY")
print("="*60)
print("\nNORMAL Images:")
print(df_normal_texture.describe())

print("\n" + "="*60)
print("PNEUMONIA Images:")
print(df_pneumonia_texture.describe())


## Visualize Texture Features Comparison
> Compare Texture features between classes

In [ ]:
texture_features = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, feature in enumerate(texture_features):
    ax = axes[idx]
    
    normal_values = df_normal_texture[feature].values
    pneumonia_values = df_pneumonia_texture[feature].values
    
    data_to_plot = [normal_values, pneumonia_values]
    bp = ax.boxplot(data_to_plot, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
    
    for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    ax.set_ylabel(feature.capitalize(), fontsize=11)
    ax.set_title(f'{feature.capitalize()} Comparison', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

# Remove extra subplot
axes[-1].axis('off')

plt.tight_layout()
plt.savefig('texture_features_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Texture features comparison saved as 'texture_features_comparison.png'")



## Image Quality Assessment - Blur Detection
> Detect Potential blur in images using laplacian variance

In [ ]:
def calculate_blur_score(image):
    """Calculate blur score using Laplacian variance"""
    return cv2.Laplacian(image, cv2.CV_64F).var()

print("\nCalculating blur scores...")

normal_blur_scores = [calculate_blur_score(img) for img in normal_samples]
pneumonia_blur_scores = [calculate_blur_score(img) for img in pneumonia_samples]

print("\n" + "="*60)
print("IMAGE QUALITY - BLUR DETECTION")
print("="*60)
print(f"\nNORMAL images:")
print(f"  Mean blur score: {np.mean(normal_blur_scores):.2f}")
print(f"  Min blur score: {np.min(normal_blur_scores):.2f}")
print(f"  Max blur score: {np.max(normal_blur_scores):.2f}")

print(f"\nPNEUMONIA images:")
print(f"  Mean blur score: {np.mean(pneumonia_blur_scores):.2f}")
print(f"  Min blur score: {np.min(pneumonia_blur_scores):.2f}")
print(f"  Max blur score: {np.max(pneumonia_blur_scores):.2f}")

print("\nNote: Lower scores indicate more blur")

# Visualize blur scores
fig, ax = plt.subplots(figsize=(12, 6))
data_to_plot = [normal_blur_scores, pneumonia_blur_scores]
bp = ax.boxplot(data_to_plot, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('Blur Score (Laplacian Variance)', fontsize=12)
ax.set_title('Image Sharpness Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.savefig('blur_detection.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Blur detection saved as 'blur_detection.png'")


## Image Entropy ANalysis
> Measure Information content / randomness in images

In [ ]:
from scipy.stats import entropy

def calculate_entropy(image):
    """Calculate Shannon entropy of an image"""
    hist, _ = np.histogram(image.flatten(), bins=256, range=(0, 256))
    hist = hist / hist.sum()  # Normalize
    hist = hist[hist > 0]  # Remove zeros
    return entropy(hist, base=2)

print("\nCalculating image entropy...")

normal_entropy = [calculate_entropy(img) for img in normal_samples]
pneumonia_entropy = [calculate_entropy(img) for img in pneumonia_samples]

print("\n" + "="*60)
print("IMAGE ENTROPY ANALYSIS")
print("="*60)
print("\nNORMAL images:")
print(f"  Mean entropy: {np.mean(normal_entropy):.3f} bits")
print(f"  Std entropy: {np.std(normal_entropy):.3f} bits")

print("\nPNEUMONIA images:")
print(f"  Mean entropy: {np.mean(pneumonia_entropy):.3f} bits")
print(f"  Std entropy: {np.std(pneumonia_entropy):.3f} bits")

print("\nNote: Higher entropy indicates more information/complexity")

# Visualize entropy
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
ax1 = axes[0]
ax1.hist(normal_entropy, bins=15, alpha=0.6, label='NORMAL', 
         color='#3498db', edgecolor='black')
ax1.hist(pneumonia_entropy, bins=15, alpha=0.6, label='PNEUMONIA', 
         color='#e74c3c', edgecolor='black')
ax1.set_xlabel('Entropy (bits)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Image Entropy Distribution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Box plot
ax2 = axes[1]
data_to_plot = [normal_entropy, pneumonia_entropy]
bp = ax2.boxplot(data_to_plot, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('Entropy (bits)', fontsize=12)
ax2.set_title('Entropy Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('entropy_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Entropy analysis saved as 'entropy_analysis.png'")


## Frequency Domain Analysis using FFT
> Analyze Frequency content in images

In [ ]:
def analyze_frequency_content(image):
    """Analyze frequency content using FFT"""
    # Apply FFT
    f_transform = np.fft.fft2(image)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = np.abs(f_shift)
    
    # Calculate energy in different frequency bands
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2
    
    # Low frequency (center region)
    mask_size = 30
    low_freq_energy = np.sum(magnitude_spectrum[crow-mask_size:crow+mask_size, 
                                                ccol-mask_size:ccol+mask_size])
    
    # High frequency (outer regions)
    total_energy = np.sum(magnitude_spectrum)
    high_freq_energy = total_energy - low_freq_energy
    
    return {
        'low_freq': low_freq_energy,
        'high_freq': high_freq_energy,
        'ratio': low_freq_energy / (high_freq_energy + 1e-10)
    }

print("\nAnalyzing frequency content...")

normal_freq = [analyze_frequency_content(img) for img in normal_samples[:10]]
pneumonia_freq = [analyze_frequency_content(img) for img in pneumonia_samples[:10]]

df_normal_freq = pd.DataFrame(normal_freq)
df_pneumonia_freq = pd.DataFrame(pneumonia_freq)

print("\n" + "="*60)
print("FREQUENCY DOMAIN ANALYSIS")
print("="*60)
print("\nNORMAL Images - Frequency Ratios (Low/High):")
print(df_normal_freq['ratio'].describe())

print("\nPNEUMONIA Images - Frequency Ratios (Low/High):")
print(df_pneumonia_freq['ratio'].describe())


## Visualize FFT Examples
> Show Frequency domain representaion

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx in range(2):
    # Original NORMAL
    ax = axes[0, idx*2]
    ax.imshow(normal_samples[idx], cmap='gray')
    ax.set_title(f'NORMAL {idx+1} - Spatial', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # FFT NORMAL
    ax = axes[0, idx*2 + 1]
    f_transform = np.fft.fft2(normal_samples[idx])
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)
    ax.imshow(magnitude_spectrum, cmap='hot')
    ax.set_title(f'NORMAL {idx+1} - Frequency', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Original PNEUMONIA
    ax = axes[1, idx*2]
    ax.imshow(pneumonia_samples[idx], cmap='gray')
    ax.set_title(f'PNEUMONIA {idx+1} - Spatial', fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # FFT PNEUMONIA
    ax = axes[1, idx*2 + 1]
    f_transform = np.fft.fft2(pneumonia_samples[idx])
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)
    ax.imshow(magnitude_spectrum, cmap='hot')
    ax.set_title(f'PNEUMONIA {idx+1} - Frequency', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('fft_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ FFT analysis saved as 'fft_analysis.png'")


## Create Feature Summary
> Compile all Extracted features into a summary DataFrame

In [ ]:
print("\n" + "="*60)
print("CREATING COMPREHENSIVE FEATURE SUMMARY")
print("="*60)

# Compile all features
feature_summary = []

for idx, (normal_img, pneumonia_img) in enumerate(zip(normal_samples, pneumonia_samples)):
    # NORMAL features
    feature_summary.append({
        'class': 'NORMAL',
        'sample_id': idx,
        'brightness': calculate_brightness(normal_img),
        'contrast': calculate_contrast(normal_img),
        'entropy': calculate_entropy(normal_img),
        'blur_score': calculate_blur_score(normal_img),
        'edge_density': calculate_edge_density(normal_img),
        **calculate_texture_features(normal_img)
    })
    
    # PNEUMONIA features
    feature_summary.append({
        'class': 'PNEUMONIA',
        'sample_id': idx,
        'brightness': calculate_brightness(pneumonia_img),
        'contrast': calculate_contrast(pneumonia_img),
        'entropy': calculate_entropy(pneumonia_img),
        'blur_score': calculate_blur_score(pneumonia_img),
        'edge_density': calculate_edge_density(pneumonia_img),
        **calculate_texture_features(pneumonia_img)
    })

df_features = pd.DataFrame(feature_summary)

print("\nFeature Summary DataFrame:")
print(df_features.head(10))
print(f"\nShape: {df_features.shape}")
print(f"\nColumns: {df_features.columns.tolist()}")

# Save to CSV
df_features.to_csv('feature_summary.csv', index=False)
print("\n✓ Feature summary saved as 'feature_summary.csv'")



## Statistical Summary of Features
> Perform Statistical tests to identify significant differences

In [ ]:
from scipy import stats

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*60)

feature_cols = ['brightness', 'contrast', 'entropy', 'blur_score', 'edge_density',
                'contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']

print("\nT-Test Results (NORMAL vs PNEUMONIA):")
print("-" * 60)

statistical_results = []

for feature in feature_cols:
    normal_values = df_features[df_features['class'] == 'NORMAL'][feature].values
    pneumonia_values = df_features[df_features['class'] == 'PNEUMONIA'][feature].values
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(normal_values, pneumonia_values)
    
    # Effect size (Cohen's d)
    cohens_d = (np.mean(normal_values) - np.mean(pneumonia_values)) / \
               np.sqrt((np.std(normal_values)**2 + np.std(pneumonia_values)**2) / 2)
    
    statistical_results.append({
        'Feature': feature,
        'T-Statistic': t_stat,
        'P-Value': p_value,
        'Cohens_D': cohens_d,
        'Significant': 'Yes' if p_value < 0.05 else 'No'
    })
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    print(f"{feature:20s} | t={t_stat:7.3f} | p={p_value:.4f} | d={cohens_d:6.3f} | {significance}")

df_stats = pd.DataFrame(statistical_results)
df_stats.to_csv('statistical_tests.csv', index=False)
print("\n✓ Statistical tests saved as 'statistical_tests.csv'")

## Correlation Analysis
> Purpose: Analyze correlations between different features

In [ ]:

print("\n" + "="*60)
print("FEATURE CORRELATION ANALYSIS")
print("="*60)

# Calculate correlation matrix
feature_columns = df_features.select_dtypes(include=[np.number]).columns.drop(['sample_id'])
correlation_matrix = df_features[feature_columns].corr()

print("\nCorrelation Matrix:")
print(correlation_matrix)

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation matrix saved as 'correlation_matrix.png'")


## Principal Component Analysis (PCA)
> Reduce dimensiontality and visualize feature space

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("\n" + "="*60)
print("PRINCIPAL COMPONENT ANALYSIS")
print("="*60)

# Prepare data
X = df_features[feature_columns].values
y = df_features['class'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print("\nExplained Variance Ratio:")
for idx, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f"  PC{idx+1}: {var:.4f} ({var*100:.2f}%)")

print("\nCumulative Explained Variance:")
cumsum = np.cumsum(pca.explained_variance_ratio_)
for idx in [0, 1, 2, 4, 9]:
    if idx < len(cumsum):
        print(f"  First {idx+1} components: {cumsum[idx]:.4f} ({cumsum[idx]*100:.2f}%)")


## Visualize PCA results
> plot PCA scatter plt and explained variance


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Scree plot
ax1 = axes[0]
ax1.bar(range(1, len(pca.explained_variance_ratio_) + 1), 
        pca.explained_variance_ratio_, alpha=0.6, color='#3498db', edgecolor='black')
ax1.plot(range(1, len(pca.explained_variance_ratio_) + 1), 
         np.cumsum(pca.explained_variance_ratio_), 
         'ro-', linewidth=2, markersize=8)
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Scree Plot - PCA Explained Variance', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(['Cumulative', 'Individual'], fontsize=10)

# PCA scatter plot
ax2 = axes[1]
for class_name, color in zip(['NORMAL', 'PNEUMONIA'], ['#3498db', '#e74c3c']):
    mask = y == class_name
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               label=class_name, alpha=0.6, s=50, color=color, edgecolor='black')

ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax2.set_title('PCA: First Two Principal Components', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ PCA analysis saved as 'pca_analysis.png'")


## Feature IMportance using Random Forest
> Identify most discriminative features

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("\n" + "="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_scaled, y_encoded)

# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nTop 10 Most Important Features:")
print("-" * 60)
for i in range(min(10, len(indices))):
    idx = indices[i]
    print(f"{i+1:2d}. {feature_columns[idx]:20s} : {importances[idx]:.4f}")

# Visualize feature importances
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(feature_columns))
ax.barh(y_pos, importances[indices], color='#3498db', alpha=0.7, edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_columns[i] for i in indices], fontsize=10)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature importance saved as 'feature_importance.png'")


## Data Augmentation Visualization
> Show Potential augmentation techniques for training

In [ ]:
from scipy.ndimage import rotate, zoom

def augment_image(image):
    """Apply various augmentation techniques"""
    augmented = {}
    
    # Original
    augmented['Original'] = image
    
    # Rotation
    augmented['Rotated 15°'] = rotate(image, 15, reshape=False, mode='nearest')
    
    # Horizontal flip
    augmented['Flipped'] = np.fliplr(image)
    
    # Brightness adjustment
    augmented['Brighter'] = np.clip(image * 1.3, 0, 255).astype(np.uint8)
    
    # Zoom
    zoomed = zoom(image, 1.2)
    h, w = image.shape
    start_h = (zoomed.shape[0] - h) // 2
    start_w = (zoomed.shape[1] - w) // 2
    augmented['Zoomed'] = zoomed[start_h:start_h+h, start_w:start_w+w]
    
    # Add noise
    noise = np.random.normal(0, 10, image.shape)
    augmented['With Noise'] = np.clip(image + noise, 0, 255).astype(np.uint8)
    
    return augmented

print("\n" + "="*60)
print("DATA AUGMENTATION EXAMPLES")
print("="*60)

# Apply augmentation to sample images
sample_normal = normal_samples[0]
sample_pneumonia = pneumonia_samples[0]

aug_normal = augment_image(sample_normal)
aug_pneumonia = augment_image(sample_pneumonia)

# Visualize augmentations
fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for idx, (aug_name, aug_img) in enumerate(aug_normal.items()):
    ax = axes[0, idx]
    ax.imshow(aug_img, cmap='gray')
    ax.set_title(f'NORMAL - {aug_name}', fontsize=10, fontweight='bold')
    ax.axis('off')

for idx, (aug_name, aug_img) in enumerate(aug_pneumonia.items()):
    ax = axes[1, idx]
    ax.imshow(aug_img, cmap='gray')
    ax.set_title(f'PNEUMONIA - {aug_name}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Augmentation examples saved as 'data_augmentation.png'")


## Final Summary Report
> Generate a comprehensive summary of all findings

In [ ]:
print("\n" + "="*80)
print(" " * 25 + "FINAL EDA SUMMARY REPORT")
print("="*80)

print("\n1. DATASET COMPOSITION")
print("-" * 80)
print(f"   Total Images: {sum(train_counts.values()) + sum(test_counts.values()) + sum(val_counts.values())}")
print(f"   Training Set: {sum(train_counts.values())} images")
print(f"      - NORMAL: {train_counts['NORMAL']} ({train_counts['NORMAL']/sum(train_counts.values())*100:.1f}%)")
print(f"      - PNEUMONIA: {train_counts['PNEUMONIA']} ({train_counts['PNEUMONIA']/sum(train_counts.values())*100:.1f}%)")
print(f"   Test Set: {sum(test_counts.values())} images")
print(f"      - NORMAL: {test_counts['NORMAL']} ({test_counts['NORMAL']/sum(test_counts.values())*100:.1f}%)")
print(f"      - PNEUMONIA: {test_counts['PNEUMONIA']} ({test_counts['PNEUMONIA']/sum(test_counts.values())*100:.1f}%)")
print(f"   Validation Set: {sum(val_counts.values())} images")
print(f"      - NORMAL: {val_counts['NORMAL']} ({val_counts['NORMAL']/sum(val_counts.values())*100:.1f}%)")
print(f"      - PNEUMONIA: {val_counts['PNEUMONIA']} ({val_counts['PNEUMONIA']/sum(val_counts.values())*100:.1f}%)")

print("\n2. CLASS IMBALANCE")
print("-" * 80)
print(f"   Training Set Imbalance Ratio: {train_ratio:.2f}:1")
print(f"   ⚠ Recommendation: Use class weights or oversampling techniques")

print("\n3. IMAGE CHARACTERISTICS")
print("-" * 80)
print(f"   Average Image Dimensions:")
print(f"      - Width: {df_metadata['width'].mean():.0f} px")
print(f"      - Height: {df_metadata['height'].mean():.0f} px")
print(f"   Average File Size: {df_metadata['file_size_kb'].mean():.2f} KB")
print(f"   Most Common Format: {df_metadata['format'].mode()[0]}")

print("\n4. PIXEL INTENSITY ANALYSIS")
print("-" * 80)
print(f"   NORMAL images - Mean intensity: {normal_mean_intensity:.2f}")
print(f"   PNEUMONIA images - Mean intensity: {pneumonia_mean_intensity:.2f}")
print(f"   Difference: {abs(normal_mean_intensity - pneumonia_mean_intensity):.2f}")

print("\n5. KEY DISCRIMINATIVE FEATURES")
print("-" * 80)
print("   Top 5 Most Important Features:")
for i in range(min(5, len(indices))):
    idx = indices[i]
    print(f"      {i+1}. {feature_columns[idx]}: {importances[idx]:.4f}")

print("\n6. STATISTICAL SIGNIFICANCE")
print("-" * 80)
significant_features = df_stats[df_stats['Significant'] == 'Yes']
print(f"   {len(significant_features)} out of {len(df_stats)} features show significant differences")
print("   Significant features:")
for _, row in significant_features.head(5).iterrows():
    print(f"      - {row['Feature']}: p={row['P-Value']:.4f}")

print("\n7. RECOMMENDATIONS FOR MODEL TRAINING")
print("-" * 80)
print("   ✓ Address class imbalance using:")
print("      - Class weights in loss function")
print("      - Oversampling minority class (NORMAL)")
print("      - Data augmentation (rotation, flip, brightness, zoom)")
print("   ✓ Image preprocessing steps:")
print("      - Resize to consistent dimensions (e.g., 224x224 or 299x299)")
print("      - Normalize pixel values (0-1 range or standardization)")
print("      - Apply histogram equalization for contrast enhancement")
print("   ✓ Consider transfer learning with pre-trained models:")
print("      - ResNet, VGG, DenseNet, or EfficientNet")
print("      - Fine-tune on this specific dataset")
print("   ✓ Evaluation metrics:")
print("      - Use F1-score, precision, recall (not just accuracy)")
print("      - ROC-AUC for binary classification")
print("      - Confusion matrix analysis")

print("\n8. FILES GENERATED")
print("-" * 80)
generated_files = [
    'class_distribution_bars.png',
    'class_distribution_pies.png',
    'image_dimensions_analysis.png',
    'file_size_analysis.png',
    'pixel_intensity_distribution.png',
    'sample_images_grid.png',
    'image_statistics_comparison.png',
    'edge_density_analysis.png',
    'edge_detection_examples.png',
    'histogram_equalization.png',
    'brightness_distribution.png',
    'contrast_analysis.png',
    'texture_features_comparison.png',
    'blur_detection.png',
    'entropy_analysis.png',
    'fft_analysis.png',
    'feature_summary.csv',
    'statistical_tests.csv',
    'correlation_matrix.png',
    'pca_analysis.png',
    'feature_importance.png',
    'data_augmentation.png'
]

for idx, file in enumerate(generated_files, 1):
    print(f"   {idx:2d}. {file}")

print("\n" + "="*80)
print(" " * 30 + "EDA COMPLETE!")
print("="*80)
print("\n✓ All analyses completed successfully")
print("✓ All visualizations saved")
print("✓ Ready for model development\n")



## Export all numerical Results
> Save all computed metrics for future refernce

In [ ]:
all_results = {
    'dataset_stats': {
        'total_images': sum(train_counts.values()) + sum(test_counts.values()) + sum(val_counts.values()),
        'train_images': sum(train_counts.values()),
        'test_images': sum(test_counts.values()),
        'val_images': sum(val_counts.values()),
        'train_normal': train_counts['NORMAL'],
        'train_pneumonia': train_counts['PNEUMONIA'],
        'test_normal': test_counts['NORMAL'],
        'test_pneumonia': test_counts['PNEUMONIA'],
        'val_normal': val_counts['NORMAL'],
        'val_pneumonia': val_counts['PNEUMONIA'],
        'train_imbalance_ratio': float(train_ratio),
        'test_imbalance_ratio': float(test_ratio),
        'val_imbalance_ratio': float(val_ratio)
    },
    'image_properties': {
        'mean_width': float(df_metadata['width'].mean()),
        'mean_height': float(df_metadata['height'].mean()),
        'mean_aspect_ratio': float(df_metadata['aspect_ratio'].mean()),
        'mean_file_size_kb': float(df_metadata['file_size_kb'].mean()),
        'std_width': float(df_metadata['width'].std()),
        'std_height': float(df_metadata['height'].std())
    },
    'pixel_analysis': {
        'normal_mean_intensity': float(normal_mean_intensity),
        'pneumonia_mean_intensity': float(pneumonia_mean_intensity),
        'normal_mean_contrast': float(np.mean(normal_contrast)),
        'pneumonia_mean_contrast': float(np.mean(pneumonia_contrast)),
        'normal_mean_entropy': float(np.mean(normal_entropy)),
        'pneumonia_mean_entropy': float(np.mean(pneumonia_entropy))
    },
    'quality_metrics': {
        'normal_mean_blur_score': float(np.mean(normal_blur_scores)),
        'pneumonia_mean_blur_score': float(np.mean(pneumonia_blur_scores)),
        'normal_mean_edge_density': float(np.mean(normal_edge_densities)),
        'pneumonia_mean_edge_density': float(np.mean(pneumonia_edge_densities))
    },
    'pca_results': {
        'pc1_variance': float(pca.explained_variance_ratio_[0]),
        'pc2_variance': float(pca.explained_variance_ratio_[1]),
        'pc1_pc2_cumulative': float(pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1])
    }
}

import json
with open('eda_numerical_results.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print("✓ All numerical results saved as 'eda_numerical_results.json'")



# DashBoard

In [ ]:
fig = plt.figure(figsize=(20, 24))
gs = fig.add_gridspec(6, 3, hspace=0.4, wspace=0.3)

# Title
fig.suptitle('CHEST X-RAY PNEUMONIA DETECTION - EDA DASHBOARD', 
             fontsize=20, fontweight='bold', y=0.995)

# 1. Class Distribution
ax1 = fig.add_subplot(gs[0, 0])
splits_data = [train_counts, test_counts, val_counts]
x = np.arange(len(CLASSES))
width = 0.25
for idx, (split_name, counts) in enumerate(zip(['Train', 'Test', 'Val'], splits_data)):
    ax1.bar(x + idx*width, counts.values(), width, label=split_name, alpha=0.8)
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.set_title('Class Distribution Across Splits', fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(CLASSES)
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Imbalance Ratio
ax2 = fig.add_subplot(gs[0, 1])
ratios = [train_ratio, test_ratio, val_ratio]
colors_bar = ['#e74c3c', '#f39c12', '#3498db']
bars = ax2.bar(['Train', 'Test', 'Val'], ratios, color=colors_bar, alpha=0.7, edgecolor='black')
ax2.axhline(y=1, color='green', linestyle='--', linewidth=2, label='Balanced (1:1)')
ax2.set_ylabel('Ratio (PNEUMONIA:NORMAL)')
ax2.set_title('Class Imbalance Ratio', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Image Dimensions
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(df_metadata['width'], df_metadata['height'], alpha=0.3, s=10)
ax3.set_xlabel('Width (px)')
ax3.set_ylabel('Height (px)')
ax3.set_title('Image Dimensions Distribution', fontweight='bold')
ax3.grid(alpha=0.3)

# 4. Brightness Comparison
ax4 = fig.add_subplot(gs[1, 0])
data_box = [normal_brightness, pneumonia_brightness]
bp = ax4.boxplot(data_box, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax4.set_ylabel('Brightness')
ax4.set_title('Brightness Comparison', fontweight='bold')
ax4.grid(alpha=0.3)

# 5. Contrast Comparison
ax5 = fig.add_subplot(gs[1, 1])
data_box = [normal_contrast, pneumonia_contrast]
bp = ax5.boxplot(data_box, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax5.set_ylabel('Contrast (Std Dev)')
ax5.set_title('Contrast Comparison', fontweight='bold')
ax5.grid(alpha=0.3)

# 6. Entropy Comparison
ax6 = fig.add_subplot(gs[1, 2])
data_box = [normal_entropy, pneumonia_entropy]
bp = ax6.boxplot(data_box, labels=['NORMAL', 'PNEUMONIA'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax6.set_ylabel('Entropy (bits)')
ax6.set_title('Entropy Comparison', fontweight='bold')
ax6.grid(alpha=0.3)

# 7. Sample NORMAL Images
for i in range(3):
    ax = fig.add_subplot(gs[2, i])
    ax.imshow(normal_samples[i], cmap='gray')
    ax.set_title(f'NORMAL Sample {i+1}', fontweight='bold', color='blue')
    ax.axis('off')

# 8. Sample PNEUMONIA Images
for i in range(3):
    ax = fig.add_subplot(gs[3, i])
    ax.imshow(pneumonia_samples[i], cmap='gray')
    ax.set_title(f'PNEUMONIA Sample {i+1}', fontweight='bold', color='red')
    ax.axis('off')

# 9. Feature Importance
ax9 = fig.add_subplot(gs[4, :2])
top_n = min(10, len(indices))  # Ensure we don't exceed available features
top_indices = indices[:top_n]
y_pos = np.arange(top_n)
ax9.barh(y_pos, importances[top_indices], color='#3498db', alpha=0.7, edgecolor='black')
ax9.set_yticks(y_pos)
ax9.set_yticklabels([feature_columns[i] for i in top_indices], fontsize=9)
ax9.set_xlabel('Importance')
ax9.set_title(f'Top {top_n} Feature Importance (Random Forest)', fontweight='bold')
ax9.grid(alpha=0.3)
ax9.invert_yaxis()

# 10. PCA Scatter
ax10 = fig.add_subplot(gs[4, 2])
for class_name, color in zip(['NORMAL', 'PNEUMONIA'], ['#3498db', '#e74c3c']):
    mask = y == class_name
    ax10.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                label=class_name, alpha=0.5, s=30, color=color)
ax10.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=9)
ax10.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=9)
ax10.set_title('PCA Projection', fontweight='bold')
ax10.legend(fontsize=8)
ax10.grid(alpha=0.3)

# 11. Summary Statistics Text
ax11 = fig.add_subplot(gs[5, :])
ax11.axis('off')
summary_text = f"""
KEY FINDINGS & RECOMMENDATIONS:

• Dataset: {sum(train_counts.values()) + sum(test_counts.values()) + sum(val_counts.values())} total images | Train: {sum(train_counts.values())} | Test: {sum(test_counts.values())} | Val: {sum(val_counts.values())}

• Class Imbalance: Training set has {train_ratio:.2f}:1 ratio (PNEUMONIA:NORMAL) - CRITICAL: Use class weights or oversampling

• Image Properties: Avg dimensions {df_metadata['width'].mean():.0f}×{df_metadata['height'].mean():.0f} px | Avg file size: {df_metadata['file_size_kb'].mean():.1f} KB

• Pixel Analysis: PNEUMONIA images show {"higher" if pneumonia_mean_intensity > normal_mean_intensity else "lower"} average brightness ({pneumonia_mean_intensity:.1f} vs {normal_mean_intensity:.1f})

• Discriminative Features: {len(significant_features)} statistically significant features identified (p < 0.05)

• Top 3 Important Features: {feature_columns[indices[0]]}, {feature_columns[indices[1]]}, {feature_columns[indices[2]]}

MODELING RECOMMENDATIONS:
✓ Use transfer learning (ResNet50, DenseNet121, or EfficientNet) with ImageNet weights
✓ Apply data augmentation: rotation (±15°), flip, brightness adjustment, zoom
✓ Implement class weights or oversampling to handle imbalance
✓ Optimize for RECALL on PNEUMONIA class (minimize false negatives)
✓ Use F1-score, precision, recall, and ROC-AUC for evaluation (not just accuracy)
✓ Resize images to 224×224 or 299×299, normalize pixels, apply histogram equalization
"""

ax11.text(0.05, 0.5, summary_text, fontsize=10, verticalalignment='center',
          family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.savefig('eda_comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comprehensive dashboard saved as 'eda_comprehensive_dashboard.png'")



## Final Cleanup and summary


In [ ]:
print("\n" + "="*80)
print(" " * 25 + "EDA SUCCESSFULLY COMPLETED!")
print("="*80)

print("\n📊 GENERATED FILES SUMMARY:")
print("-" * 80)

file_categories = {
    '📈 Visualizations (PNG)': [
        'class_distribution_bars.png',
        'class_distribution_pies.png',
        'image_dimensions_analysis.png',
        'file_size_analysis.png',
        'pixel_intensity_distribution.png',
        'sample_images_grid.png',
        'image_statistics_comparison.png',
        'edge_density_analysis.png',
        'edge_detection_examples.png',
        'histogram_equalization.png',
        'brightness_distribution.png',
        'contrast_analysis.png',
        'texture_features_comparison.png',
        'blur_detection.png',
        'entropy_analysis.png',
        'fft_analysis.png',
        'correlation_matrix.png',
        'pca_analysis.png',
        'feature_importance.png',
        'data_augmentation.png',
        'eda_comprehensive_dashboard.png'
    ],
    '📄 Data Files (CSV/JSON)': [
        'feature_summary.csv',
        'statistical_tests.csv',
        'eda_summary_table.csv',
        'eda_numerical_results.json'
    ],
    '📝 Documentation (TXT)': [
        'modeling_recommendations.txt'
    ]
}

for category, files in file_categories.items():
    print(f"\n{category}:")
    for file in files:
        print(f"   ✓ {file}")

print("\n" + "="*80)
print("🚀 NEXT STEPS:")
print("="*80)
print("""
1. Review all generated visualizations to understand data patterns
2. Read 'modeling_recommendations.txt' for detailed guidance
3. Start implementing data preprocessing pipeline
4. Begin with baseline model (ResNet50 + transfer learning)
5. Experiment with different architectures and hyperparameters
6. Focus on maximizing RECALL for PNEUMONIA class
7. Use Grad-CAM for model interpretability

💡 REMEMBER: This is a medical imaging task - false negatives (missing pneumonia)
   are more critical than false positives. Optimize accordingly!
""")